# TFT Training-Window Robustness

Training-length robustness check for the TFT. Supports the main experiment in validation_regime_experiment.ipynb.

The question: does the calm-versus-stress gap survive if Optuna re-selects the TFT hyperparameters separately at each training-window length, not just under one fixed window?

For each training start it runs a fresh 30-trial Optuna search on the calm window and another on the stress window, then reports the calm-versus-stress high-volatility QLIKE gap at the 33/67 and 25/75 thresholds. The validation windows and the test period are fixed, exactly as in the main experiment.

Training starts:
- 2000_full: 2000-08-01, about 3122 days.
- 2004_b: 2004-01-01, about 2265 days.
- 2005_short: 2005-01-01, about 2013 days.

This is a long run. Three windows times two selections times thirty trials is 180 full TFT fits. A GPU is strongly recommended.

Data is pulled from Yahoo Finance at run time, the same source as the main experiment.

In [ ]:
# ============================================================
# TFT OPTUNA RE-SELECTION SWEEP  (training-window robustness, full re-tuning)
#
# Purpose: answer the objection "would the TFT gap survive if Optuna re-selected
# hyperparameters per training window, not just under a frozen config?"
# This mirrors the LSTM re-selection sweep: for each training window we run a
# FRESH Optuna search (30 trials, matching Paper 1) selecting on the calm window,
# and another selecting on the stress window, then compute the calm-vs-stress
# high-vol QLIKE gap from the two RE-SELECTED configs.
#
# Windows (train END fixed at 2013-01-01; only START varies):
#   2000_full  : 2000-08-01  (~3122 days)
#   2004_b     : 2004-01-01  (~2265 days)
#   2005_short : 2005-01-01  (~2013 days)
# (2003 is NOT run -- already known unstable, excluded from the paper.)
#
# Validation windows are FIXED (calm 2013-15, stress 2015-17), exactly as Paper 1.
# Test = 2020+. Thresholds 33/67 and 25/75.
#
# This is the OVERNIGHT job: 3 windows x 2 selections x 30 trials = 180 TFT Optuna
# trials, each a full TFT fit. Slow. Run with margin.
#
# Self-contained: paste as one cell into a fresh environment.
# ============================================================

import numpy as np
import pandas as pd
import torch
import yfinance as yf
import optuna
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import RMSE
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_TRIALS = 30          # <-- matches Paper 1's TFT budget (confirmed from the Optuna log)
np.random.seed(SEED)
torch.manual_seed(SEED)
MAX_PRED = 1

# ------------------------------------------------------------
# DATA (Garman-Klass realized variance, time-indexed frame for the TFT)
# ------------------------------------------------------------
sp500 = yf.download("^GSPC", start="2000-08-01", end="2025-01-22",
                    progress=False, auto_adjust=False)
if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)
sp500 = sp500[["Open", "High", "Low", "Close"]].copy()
sp500["rv_gk"] = (0.5 * (np.log(sp500["High"] / sp500["Low"])) ** 2
                  - (2 * np.log(2) - 1) * (np.log(sp500["Close"] / sp500["Open"])) ** 2)
sp500 = sp500.dropna()

sp500_ti = sp500.reset_index().rename(columns={"Date": "date"})
sp500_ti["date"] = pd.to_datetime(sp500_ti["date"])
sp500_ti["time_idx"] = np.arange(len(sp500_ti))
sp500_ti["group"] = "SP500"
sp500_ti = sp500_ti[["date", "time_idx", "group", "rv_gk"]]
print(f"Total obs: {len(sp500_ti)} ({sp500_ti['date'].min().date()} -> {sp500_ti['date'].max().date()})")

# ------------------------------------------------------------
# Window constants
# ------------------------------------------------------------
TRAIN_END    = "2013-01-01"
CALM_START, CALM_END     = "2013-01-01", "2015-01-01"
STRESS_START, STRESS_END = "2015-01-01", "2017-01-01"
TEST_START   = "2020-01-01"

TRAIN_STARTS = {
    "2000_full":  "2000-08-01",
    "2004_b":     "2004-01-01",
    "2005_short": "2005-01-01",
}

# ------------------------------------------------------------
# QLIKE
# ------------------------------------------------------------
def calculate_qlike(actual, predicted):
    actual = np.array(actual).flatten(); predicted = np.array(predicted).flatten()
    eps = 1e-10
    predicted = np.maximum(predicted, eps); actual = np.maximum(actual, eps)
    return np.mean(actual / predicted - np.log(actual / predicted) - 1)

# ------------------------------------------------------------
# tft_build with TRAIN-START floor (otherwise verbatim from your pipeline)
# ------------------------------------------------------------
def tft_build_ts(train_start, val_start, val_end, max_encoder):
    train_start_idx = sp500_ti[sp500_ti["date"] >= train_start]["time_idx"].min()
    train_end_idx   = sp500_ti[sp500_ti["date"] < TRAIN_END]["time_idx"].max()
    val_end_idx     = sp500_ti[sp500_ti["date"] < val_end]["time_idx"].max()
    val_start_idx   = sp500_ti[sp500_ti["date"] < val_start]["time_idx"].max()

    train_df = sp500_ti[(sp500_ti["time_idx"] >= train_start_idx) &
                        (sp500_ti["time_idx"] <= train_end_idx)]

    training = TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="rv_gk", group_ids=["group"],
        max_encoder_length=max_encoder, max_prediction_length=MAX_PRED,
        static_categoricals=["group"],
        time_varying_known_reals=["time_idx"],
        time_varying_unknown_reals=["rv_gk"],
        target_normalizer=GroupNormalizer(groups=["group"], transformation="softplus"),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
    )
    validation = TimeSeriesDataSet.from_dataset(
        training,
        sp500_ti[(sp500_ti["time_idx"] > val_start_idx - max_encoder) &
                 (sp500_ti["time_idx"] <= val_end_idx)],
        stop_randomization=True,
    )
    test_start_idx = sp500_ti[sp500_ti["date"] < TEST_START]["time_idx"].max()
    test_ds = TimeSeriesDataSet.from_dataset(
        training,
        sp500_ti[sp500_ti["time_idx"] > test_start_idx - max_encoder],
        stop_randomization=True,
    )
    return training, validation, test_ds

# ------------------------------------------------------------
# tft_train_eval (verbatim selection logic; train_start added)
# ------------------------------------------------------------
def tft_train_eval_ts(cfg, train_start, val_start, val_end, epochs=100, patience=15, seed=SEED):
    pl.seed_everything(seed, workers=True)
    training, validation, test_ds = tft_build_ts(train_start, val_start, val_end,
                                                 cfg["max_encoder_length"])
    tdl = training.to_dataloader(train=True,  batch_size=cfg["batch_size"], num_workers=0)
    vdl = validation.to_dataloader(train=False, batch_size=cfg["batch_size"], num_workers=0)
    xdl = test_ds.to_dataloader(train=False, batch_size=cfg["batch_size"], num_workers=0)

    model = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=cfg["learning_rate"], hidden_size=cfg["hidden_size"],
        attention_head_size=cfg["attention_head_size"], dropout=cfg["dropout"],
        hidden_continuous_size=cfg["hidden_continuous_size"],
        lstm_layers=cfg.get("lstm_layers", 1),
        output_size=1, loss=RMSE(), log_interval=-1, reduce_on_plateau_patience=3,
    )
    es = EarlyStopping(monitor="val_loss", patience=patience, mode="min")
    trainer = pl.Trainer(max_epochs=epochs, accelerator="auto", devices=1,
                         gradient_clip_val=0.1, callbacks=[es],
                         enable_progress_bar=False, enable_model_summary=False, logger=False)
    trainer.fit(model, tdl, vdl)

    vpred = model.predict(vdl).cpu().numpy().astype(np.float64)
    vact  = torch.cat([y[0] for x, y in iter(vdl)]).cpu().numpy().astype(np.float64)
    val_qlike = calculate_qlike(vact, vpred)
    tpred = model.predict(xdl).cpu().numpy().astype(np.float64).flatten()
    return tpred, float(val_qlike)

# ------------------------------------------------------------
# Optuna objective factory (verbatim search space; train_start threaded in)
# ------------------------------------------------------------
def tft_objective_factory(train_start, val_start, val_end):
    def objective(trial):
        cfg = {
            "hidden_size": trial.suggest_categorical("hidden_size", [16, 32, 48, 64]),
            "attention_head_size": trial.suggest_categorical("attention_head_size", [1, 2, 4]),
            "dropout": trial.suggest_float("dropout", 0.05, 0.4, step=0.05),
            "hidden_continuous_size": trial.suggest_categorical("hidden_continuous_size", [8, 16, 32]),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
            "max_encoder_length": trial.suggest_categorical("max_encoder_length", [10, 22, 44]),
            "lstm_layers": trial.suggest_int("lstm_layers", 1, 2),
            "batch_size": 32,
        }
        try:
            _, val_qlike = tft_train_eval_ts(cfg, train_start, val_start, val_end,
                                             epochs=100, patience=15)
        except Exception as e:
            print(f"Trial {trial.number} failed: {e}")
            raise optuna.TrialPruned()
        return val_qlike
    return objective

# ------------------------------------------------------------
# Evaluation: QLIKE per-day + HLN-corrected DM
# ------------------------------------------------------------
def qlike_vec(a, f):
    eps = 1e-10
    a = np.maximum(a, eps); f = np.maximum(f, eps)
    return a / f - np.log(a / f) - 1

def dm_test(a, f_calm, f_stress, mask, h_step=1):
    d = qlike_vec(a, f_calm)[mask] - qlike_vec(a, f_stress)[mask]
    nm = len(d); dbar = d.mean()
    h = max(h_step - 1, int(np.floor(4 * (nm / 100.0) ** (2.0 / 9.0))))
    g0 = np.sum((d - dbar) ** 2) / nm
    var = g0
    for k in range(1, h + 1):
        ck = np.sum((d[k:] - dbar) * (d[:-k] - dbar)) / nm
        var += 2 * (1 - k / (h + 1)) * ck
    dm = dbar / np.sqrt(var / nm)
    hln = dm * np.sqrt((nm + 1 - 2 * h_step + h_step * (h_step - 1) / nm) / nm)
    return dbar, hln, nm

# ============================================================
# RUN: per window, Optuna-select on calm and on stress, then gap
# ============================================================
test_actual = sp500_ti[sp500_ti["date"] >= TEST_START]["rv_gk"].to_numpy(float)
results = {}

for lab, tstart in TRAIN_STARTS.items():
    n_days = int(sp500_ti[(sp500_ti["date"] >= tstart) &
                          (sp500_ti["date"] < TRAIN_END)].shape[0])
    print(f"\n{'='*70}\nTRAIN START {lab} ({tstart})  ~{n_days} days\n{'='*70}")
    results[lab] = {}

    for arm, vstart, vend in [("calm", CALM_START, CALM_END),
                              ("stress", STRESS_START, STRESS_END)]:
        print(f"\n  --- {lab}: Optuna selecting on {arm} validation ({N_TRIALS} trials) ---")
        sampler = optuna.samplers.TPESampler(seed=SEED)
        study = optuna.create_study(direction="minimize", sampler=sampler)
        study.optimize(tft_objective_factory(tstart, vstart, vend),
                       n_trials=N_TRIALS, show_progress_bar=False)
        best = dict(study.best_params); best["batch_size"] = 32
        # retrain the winning config with the fixed seed to get the test forecast
        tpred, vq = tft_train_eval_ts(best, tstart, vstart, vend, seed=SEED)
        results[lab][arm] = {"cfg": best, "fc": tpred, "val_qlike": vq}
        print(f"  {arm:6s} best val QLIKE={study.best_value:.4f} | retrain val={vq:.4f}")
        print(f"  {arm:6s} cfg: {best}")

# ============================================================
# READ: calm-vs-stress gap from RE-SELECTED configs, both thresholds
# ============================================================
print(f"\n\n{'#'*70}\nTFT RE-SELECTION GAP (Optuna per window) -- both thresholds\n{'#'*70}")
print("Configs were re-selected by Optuna at each window; this is the apples-to-apples\nmatch to the LSTM re-selection sweep.\n")

for lab in TRAIN_STARTS:
    calm_fc   = results[lab]["calm"]["fc"]
    stress_fc = results[lab]["stress"]["fc"]
    n = min(len(test_actual), len(calm_fc), len(stress_fc))
    a, cf, sf = test_actual[-n:], calm_fc[-n:], stress_fc[-n:]
    cvq = results[lab]["calm"]["val_qlike"]; svq = results[lab]["stress"]["val_qlike"]
    print(f"--- {lab}  (re-selected val: calm={cvq:.4f}, stress={svq:.4f}; n={n}) ---")
    for lo, hi in [(33, 67), (25, 75)]:
        qlo, qhi = np.percentile(a, lo), np.percentile(a, hi)
        low, high = a <= qlo, a >= qhi
        c_hi, s_hi = qlike_vec(a, cf)[high].mean(), qlike_vec(a, sf)[high].mean()
        c_lo, s_lo = qlike_vec(a, cf)[low].mean(),  qlike_vec(a, sf)[low].mean()
        dbar, dm, nm = dm_test(a, cf, sf, high)
        print(f"  [{lo}/{hi}] HIGH: calm={c_hi:.3f} stress={s_hi:.3f} "
              f"gap={c_hi-s_hi:+.3f} DM={dm:.2f} n_high={int(high.sum())} | "
              f"LOW: calm={c_lo:.3f} stress={s_lo:.3f}")
    print()

print("DONE. If gaps are positive + significant at all three windows under full")
print("per-window Optuna re-selection, the 'frozen-config' objection is closed for the TFT.")